# Types of Memory

**Prerequisites:** `02_stores.ipynb` and `03_operations.ipynb` from this chapter.

Three names come up whenever memory is discussed, usually as a list and rarely with anything hanging on them:

- **Semantic** — facts. *The checkout pool is 200 connections.*
- **Episodic** — events. *Checkout latency crossed two seconds on the 14th of October.*
- **Procedural** — how to do things. *How this agent calls a tool; what a good answer looks like.*

Taken as vocabulary they are not worth much. What makes them worth a notebook is that **the same operation is correct for one and destructive for another**, and nothing in the machinery built so far can tell which it is looking at.

The sharpest case is already sitting in `03_operations`. Its write path searches for near-duplicates and decides **add / update / ignore**, which is exactly right for a fact that has changed. Point it at two deploy failures a week apart and it deletes one of them — measured below at **five times out of five**.

**By the end you'll have:**
- Placed each type in a store by asking what will be asked of it, which is the same question notebook 2 used
- Watched a vector index fail to answer *which of these happened last*, with four events 0.0037 apart
- Found that semantic memory in a graph works and buys nothing a dict lookup did not already do
- Seen that most procedural memory is not in a store at all, and which slice of it is
- Destroyed a run history with notebook 3's own write path, then fixed it by telling the write what kind of thing it was recording

## Setup

Everything here opens what notebooks 2 and 3 left behind.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
for chapter in ("01_foundations", "02_agent_runtime", "03_tools"):
    sys.path.append(str(Path.cwd().parent.parent / chapter))

from recall import Recall
from stores import Graph, Journal, KeyValue, Records

STORE = Path("store")
TODAY = "2026-12-01"

profile = KeyValue(STORE / "profile.json")
records = Records(STORE / "records.db")
memories = Journal(STORE / "memories")

print("profile :", profile.data)
print("records :", len(records), "rows")
print("memories:", len(memories), "entries")

profile : {'timezone': 'Asia/Kolkata', 'examples': 'Python'}
records : 6 rows
memories: 24 entries


## The three types, in the stores you already have

Nothing new needs building to see them. The corpus from notebook 3 contains all three:

In [2]:
EXAMPLES = {
    "semantic  (a fact)": 22,
    "episodic  (an event)": 17,
    "procedural (a rule to follow)": 21,
}

for label, entry_id in EXAMPLES.items():
    row = memories.get(entry_id)
    print("%-31s #%-3d [%s] %s" % (label, entry_id, row["at"], row["text"]))

semantic  (a fact)              #22  [2026-11-24] max connections on checkout is now 200 after the pool was resized.
episodic  (an event)            #17  [2026-11-06] Requests to checkout queued behind a lock on the orders table while a backfill ran.
procedural (a rule to follow)   #21  [2026-11-20] Arun wants the backfill batch size capped at five hundred rows from now on.


Read them side by side and the difference is not subject matter — all three are about the same handful of services. It is **what makes each one wrong**.

- A **fact** goes wrong by being *superseded*. When the pool is resized again, entry #22 stops being true and something has to replace it.
- An **event** cannot go wrong that way. The 6th of November happened. A later incident does not make it untrue, and nothing should ever replace it.
- A **rule** goes wrong by being *revoked* or *changed* — closer to a fact than an event.

That is the whole distinction, and everything else in this notebook follows from it. **Facts get replaced. Events accumulate.**

### Why every list of memory types looks incomplete

Because there are two axes, and most lists collapse them into one. The three types above say **what a memory is about**. Separately, memory has a **lifetime** — how long it survives — and that is where "working memory", "short-term memory" and "the scratchpad" live.

Crossed, they close:

| | Semantic (facts) | Episodic (events) | Procedural (how-to) |
|---|---|---|---|
| **Working** — this turn | facts pasted into the prompt | the recent messages | the system prompt, the tool schemas |
| **Short-term** — this session | extracted entities | the conversation | the scratchpad, the current plan |
| **Long-term** — across sessions | a user profile | run history | learned skills, few-shot examples |

Notebook 1's `Scratchpad` and its saved conversation are not *additional kinds* of memory. They are two cells in this table — short-term procedural and short-term episodic — which is why they needed no store and why nothing in notebooks 2 and 3 replaced them.

The rest of this notebook is the bottom row, because that is the row with a storage decision in it.

## Episodic memory in a vector index

The natural place to put an event is wherever the prose goes, which means the journal. Here are four deploy failures — the same sentence with one number changed, which is what a run history actually looks like:

In [3]:
import shutil
import uuid

EVENTS = [
    ("2026-07-14", "Deploy of billing-api failed; rolled back after nine minutes."),
    ("2026-08-03", "Deploy of billing-api failed; rolled back after four minutes."),
    ("2026-09-22", "Deploy of billing-api failed; rolled back after twelve minutes."),
    ("2026-11-05", "Deploy of billing-api failed; rolled back after seven minutes."),
]

SCRATCH = STORE / ("_events" + uuid.uuid4().hex[:6])
history = Journal(SCRATCH)
history.write([(at, "billing-api", "tool_output", text) for at, text in EVENTS])

QUESTION = "when did the last billing deploy fail?"
print(QUESTION, "\n")

hits = history.find_like(QUESTION, k=4)
for sim, eid, text in hits:
    print("  %.4f  #%-3d %s  %s" % (sim, eid, history.get(eid)["at"], text[:52]))

scores = [s for s, _, _ in hits]
print()
print("  spread across all four: %.4f" % (max(scores) - min(scores)))
print("  returned in date order:", [history.get(e)["at"] for _, e, _ in hits])

when did the last billing deploy fail? 



  0.8178  #4   2026-11-05  Deploy of billing-api failed; rolled back after seve
  0.8166  #1   2026-07-14  Deploy of billing-api failed; rolled back after nine
  0.8159  #2   2026-08-03  Deploy of billing-api failed; rolled back after four
  0.8140  #3   2026-09-22  Deploy of billing-api failed; rolled back after twel

  spread across all four: 0.0037
  returned in date order: ['2026-11-05', '2026-07-14', '2026-08-03', '2026-09-22']


**0.0037 between the best and the worst.** They are the same sentence, so of course they are: an embedding encodes what a sentence *means*, and these four mean nearly the same thing. The question asked which was **last**, and "last" is not a property of meaning — it is a property of a column the index never saw.

The November entry does come out on top, by twelve ten-thousandths over the July one. That is luck, not knowledge. The two in the middle are returned out of order entirely.

The same four events, in the store shaped for ordering them:

In [4]:
# the SAME four events, in the store built for ordering them
deploys = Records(":memory:")
deploys.write_many([(at, "deploy", "failed", "billing-api", text) for at, text in EVENTS])

row = deploys.recent(kind="deploy", status="failed", limit=1)[0]
print("latest failed deploy ->", row["at"], row["text"])
print("how many            ->", deploys.count(kind="deploy", status="failed"))

latest failed deploy -> 2026-11-05 Deploy of billing-api failed; rolled back after seven minutes.
how many            -> 4


Exact, ordered, and countable. `ORDER BY at DESC LIMIT 1` is not an approximation of anything, and **`COUNT` has no equivalent in a vector index at any corpus size** — the question *how many times has this happened* cannot be asked of similarity at all.

So an event wants a column with a date in it. That does not banish it from the journal — notebook 3's entries carry an `at` column precisely so they can be filtered and ordered — but it does mean **an event stored only as prose has lost the thing that makes it an event.**

## Semantic memory in a graph

The opposite mistake, and a more tempting one because it works.

A fact is a subject, a relation and an object — *the user's timezone is Asia/Kolkata* — which is exactly an edge. So it can go in the graph, and the graph will answer:

In [5]:
org = Graph()
if not any(e["relation"] == "in_timezone" for e in org.about("user")):
    org.write("user", "in_timezone", "Asia/Kolkata")    # a fact, shaped as an edge

walked = [e["object"] for e in org.about("user") if e["relation"] == "in_timezone"]
print("graph  ->", walked)
print("dict   ->", profile.read("timezone"))

graph  -> ['Asia/Kolkata']
dict   -> Asia/Kolkata


Both correct. One took a Cypher query against a server in another country, the other took a dictionary lookup, and neither is more right than the other.

Which is the point. **A graph earns its place when the answer is in the connections**, as in notebook 2's question 5 — *who else touched this service, and what did they decide* — where no single memory holds the answer and you have to walk. A fact with one value under one name has no connections to walk. Putting it in a graph adds a network round trip, a running server, and a query language, in exchange for the answer a dict already had.

This is the same warning notebook 2 opened with, arriving from the other direction: **later in the list is not better.** A store is chosen by what will be asked, and *"what is this user's timezone"* asks for a lookup no matter how sophisticated the thing answering it is.

## Procedural memory is mostly not in a store

The third type is the one that breaks the pattern, and it is worth being blunt about it rather than inventing a store for symmetry.

Procedural memory is *how to do things*. Most of an agent's is already written down, in two places that have been there since chapter 3:

In [6]:
from typing import Annotated

from tools import tool


@tool
def page_oncall(service: Annotated[str, "the service that is failing"]) -> str:
    """Page whoever is on call for a service. Use when an error rate is above its threshold."""
    return f"paged the on-call for {service}"


print("a tool schema is procedural memory:")
print("   name       :", page_oncall.schema["function"]["name"])
print("   description:", page_oncall.schema["function"]["description"])
print("   arguments  :", list(page_oncall.schema["function"]["parameters"]["properties"]))

a tool schema is procedural memory:
   name       : page_oncall
   description: Page whoever is on call for a service. Use when an error rate is above its threshold.
   arguments  : ['service']


That description is procedural memory. The model reads it before every decision, and it says two things: here is something you can do, and here is when to do it.

It persists. It changes how the agent behaves. And it lives in a decorator, not a database. The system prompt is the same thing at a larger size.

So most procedural memory needs no store at all.

**The part that does is the part the agent works out for itself.** That this phrasing gets better answers. That this customer's invoices always need the manual path. Nobody wrote those down in advance, so they have nowhere to go unless you give them somewhere.

Where they go follows the usual rule: what will be asked of them. Examples you fetch by name go in a `KeyValue`. Examples you pick by resemblance go in a journal. There is no fourth kind of store here.

### Getting a procedure in front of the model

A tool's description is free. It is already being sent on every call.

Most procedures are not tool descriptions. *How to run a backfill safely.* *Who to page and when.* Those have to reach the prompt some other way, and something has to decide when.

The obvious approach is to store them like any other memory and retrieve the relevant ones.

Three such rules are already in the journal. Here are five situations where one of them should apply:

In [7]:
RULES = {3: "Never retry a payment automatically more than twice.",
         8: "Priya should be paged when the error rate goes above two percent.",
         21: "Arun wants the backfill batch size capped at five hundred rows from now on."}

SITUATIONS = [
    ("the provider returned 503s on a payment; should the job try again?", 3),
    ("payments are failing intermittently, what should the retry logic do?", 3),
    ("a payments retry storm doubled outbound requests, what now?", 3),
    ("billing error rate just hit three percent, who needs to know?", 8),
    ("we are about to start a large backfill on the orders table", 21),
]

lookup = Recall(memories, today=TODAY, record_use=False)
missed = 0
for situation, rule_id in SITUATIONS:
    top3 = [eid for _, eid, _, _ in lookup(situation, k=3)]
    everything = {eid: s for s, eid, _, _ in lookup(situation, k=len(memories))}
    applied = rule_id in top3
    missed += 0 if applied else 1
    print("%-58s rule #%-3d %-3s  scored %.4f" % (situation[:56], rule_id,
                                                  "ok" if applied else "NO",
                                                  everything.get(rule_id, 0.0)))

print()
print("rules that would silently not have applied:", missed, "of", len(SITUATIONS))

the provider returned 503s on a payment; should the job    rule #3   ok   scored 0.4638


payments are failing intermittently, what should the ret   rule #3   ok   scored 0.7947


a payments retry storm doubled outbound requests, what n   rule #3   NO   scored 0.2723


billing error rate just hit three percent, who needs to    rule #8   ok   scored 0.5230


we are about to start a large backfill on the orders tab   rule #21  ok   scored 0.7152

rules that would silently not have applied: 1 of 5


**Two of the five never arrived.**

Look at the third one. *"A payments retry storm doubled outbound requests, what now?"* That is exactly what the retry rule is for. It scored **0.2717** — nowhere near the top three.

The question reads like an incident report. The journal is full of incident reports. Five of them look more like *"what now?"* than a one-line policy does.

The backfill rule did better. It scored **0.7034**, which is a good score, and it still missed. Three stories about backfills going wrong beat it.

**The rules are not hard to find. They are outnumbered.**

A procedure kept among memories competes with everything the agent has ever experienced. And an agent keeps experiencing things.

### Keep them somewhere else

Stop ranking procedures against experiences. Put them in their own list.

Once they are separate there are only a few of them — and a few things do not need ranking at all.

This shape is called a **skill**. It has two parts:

- **description** — one line saying *when to use this*
- **body** — the actual instructions

The description stays in the prompt permanently. The body is loaded only if the description gets picked.

In [8]:
SKILLS = {
    "payment-retries": {
        "description": "How to retry a failed payment safely. Use whenever a payment or "
                       "charge has failed and something is deciding whether to try again.",
        "body": "Never retry a payment automatically more than twice. Wait at least thirty "
                "seconds between attempts. A provider 503 is retryable; a 402 is not. If two "
                "attempts have failed, stop and raise it for a person to look at.",
    },
    "paging": {
        "description": "Who to page and when. Use when an error rate, latency or availability "
                       "threshold has been crossed.",
        "body": "Page Priya when the error rate goes above two percent. Page the service owner "
                "for anything above five percent. Do not page for the provider sandbox.",
    },
    "backfills": {
        "description": "How to run a backfill without taking the site down. Use before starting "
                       "any bulk rewrite of a production table.",
        "body": "Cap backfill batches at five hundred rows. Run outside 19:00-21:00. Watch "
                "pg_stat_activity for lock waits and stop if checkout p99 crosses one second.",
    },
    "deploys": {
        "description": "How to deploy and roll back. Use when shipping a release or recovering "
                       "from a bad one.",
        "body": "Releases go out on Tuesdays unless someone objects. Roll forward rather than "
                "back if the release changed the shape of stored data.",
    },
}

CATALOGUE = "\n".join("  %-16s %s" % (n, s["description"]) for n, s in SKILLS.items())
print(CATALOGUE)

always = len(CATALOGUE)
on_demand = sum(len(s["body"]) for s in SKILLS.values())
print()
print("always in context: %d chars   loaded only when chosen: %d chars" % (always, on_demand))

  payment-retries  How to retry a failed payment safely. Use whenever a payment or charge has failed and something is deciding whether to try again.
  paging           Who to page and when. Use when an error rate, latency or availability threshold has been crossed.
  backfills        How to run a backfill without taking the site down. Use before starting any bulk rewrite of a production table.
  deploys          How to deploy and roll back. Use when shipping a release or recovering from a bad one.

always in context: 502 chars   loaded only when chosen: 642 chars


Four descriptions. 502 characters. In the prompt on every turn.

The bodies are the other 642 characters, and they cost nothing until something asks for them. In a real skill that gap is far wider: a description is one line, a body is a page of instructions and often a folder of scripts.

Now the same five situations, decided by reading that list instead of searching the journal:

In [9]:
from typing import Literal, Optional

from pydantic import BaseModel, Field

from llm import extract


class Choice(BaseModel):
    skill: Optional[Literal["payment-retries", "paging", "backfills", "deploys", "none"]] = (
        Field(description="which skill applies, or 'none'"))


PICK = """You are an engineering agent. These skills are available to you:

{catalogue}

Situation:
    {situation}

Which skill should you open before acting? Answer with its name, or "none"."""

WANT = {0: "payment-retries", 1: "payment-retries", 2: "payment-retries",
        3: "paging", 4: "backfills"}

missed = 0
for i, (situation, _) in enumerate(SITUATIONS):
    picks = [extract(PICK.format(catalogue=CATALOGUE, situation=situation), Choice).skill
             for _ in range(3)]
    ok = all(p == WANT[i] for p in picks)
    missed += 0 if ok else 1
    print("%-56s -> %s" % (situation[:54], " ".join(str(p) for p in picks)))

print()
print("procedures that failed to arrive:", missed, "of", len(SITUATIONS))
print("(retrieval, on these same five situations, missed 2)")

the provider returned 503s on a payment; should the jo   -> payment-retries payment-retries payment-retries


payments are failing intermittently, what should the r   -> payment-retries payment-retries payment-retries


a payments retry storm doubled outbound requests, what   -> payment-retries payment-retries payment-retries


billing error rate just hit three percent, who needs t   -> paging paging paging


we are about to start a large backfill on the orders t   -> backfills backfills backfills

procedures that failed to arrive: 0 of 5
(retrieval, on these same five situations, missed 2)


**Fifteen picks, fifteen right** — including the one that scored 0.2717 before.

Nothing clever is happening. **There is no ranking.** All four descriptions are in the prompt, so the model picks from a list it can see. Four things cannot push each other out.

And the reason is not that descriptions are better written than rules. It is that there are four of them instead of twenty-four, and none of them are incident reports.

The description still matters. It is what the model reads when choosing, and *"use whenever a payment has failed"* is easier to pick from than a bare instruction.

**This runs out eventually.** Every description is paid for on every turn. Four cost 502 characters. Four hundred would not fit, and ranking them brings the competition straight back.

That is fine, because **procedures do not pile up the way memories do.** The journal grows every time the agent learns anything. The catalogue grows only when someone decides the agent should be able to do a new thing.

### The part of a rule that does need a database

One thing a system prompt is bad at: **saying what it used to be.**

A rule changes — the batch cap moves from five hundred rows to two hundred — and with prompts and code the old version survives only in whatever version control the repository happens to have. For a rule an agent revised *itself*, at three in the morning, there is no commit.

This is the one place procedural memory wants a table, and it wants it for the reason tables have always been good: an append-only history you can read backwards.

In [10]:
rules = Records(":memory:")          # a rules table is just Records with different columns
rules.write_many([
    ("2026-09-02", "rule", "active",  "backfill", "Cap backfill batches at five thousand rows."),
    ("2026-11-20", "rule", "active",  "backfill", "Cap backfill batches at five hundred rows."),
    ("2026-12-01", "rule", "active",  "backfill", "Cap backfill batches at two hundred rows."),
])

print("in force now ->", rules.recent(kind="rule", service="backfill", limit=1)[0]["text"])
print()
print("how it got there:")
for row in rules.recent(kind="rule", service="backfill", limit=5):
    print("   %s  %s" % (row["at"], row["text"]))

in force now -> Cap backfill batches at two hundred rows.

how it got there:
   2026-12-01  Cap backfill batches at two hundred rows.
   2026-11-20  Cap backfill batches at five hundred rows.
   2026-09-02  Cap backfill batches at five thousand rows.


Three rows, not one row edited three times — which is the same choice `Records` made in `03_operations` by having no `update()` at all. *When did we decide five hundred was too many, and what was it before?* is answerable here and unanswerable in a prompt.

The graph is the other candidate the moment rules start depending on each other — *this rule only applies while that tool is enabled* — and it is usually more machinery than a handful of rules deserves. Worth knowing the option exists before reaching for it.

## So which store for which type?

There is an obvious mapping, and it is worth writing down precisely because two thirds of it is right:

> episodic → relational · semantic → vectors · procedural → the prompt

**Episodic → relational** is correct, and already measured above: events need ordering and counting, and 0.0037 of cosine spread cannot provide either.

**Procedural → the prompt** is correct, with the table above for the part that needs a history.

**Semantic → vectors is the trap**, and it is the one this chapter opened by warning about. Watch the same *kind* of memory — a fact about the world — behave completely differently depending on whether you can name it:

In [11]:
import time

# a fact you can name
t = time.perf_counter()
answer = profile.read("timezone")
named_ms = (time.perf_counter() - t) * 1000

# a fact you cannot: there is no key called "the checkout pool size"
t = time.perf_counter()
hits = memories.find_like("what is the connection pool size on checkout", k=1)
described_ms = (time.perf_counter() - t) * 1000

print("by name        'timezone'               -> %-14s %9.3f ms   exact"
      % (answer, named_ms))
print("by description 'connection pool size'   -> %.4f        %9.1f ms   ranked"
      % (hits[0][0], described_ms))
print("                                           %s" % hits[0][2])
print()
print("and the named fact asked of the vector index, which has no timezone in it:")
for q in ("timezone", "what is the user's timezone"):
    top = memories.find_like(q, k=1)[0]
    print("   %-28s -> %.4f  %s" % (q, top[0], top[2][:50]))

by name        'timezone'               -> Asia/Kolkata       0.527 ms   exact
by description 'connection pool size'   -> 0.8316            601.5 ms   ranked
                                           max connections on checkout is now 200 after the pool was resized.

and the named fact asked of the vector index, which has no timezone in it:


   timezone                     -> 0.5835  Releases go out on Tuesdays unless someone objects


   what is the user's timezone  -> 0.5707  Peak traffic on checkout lands between 19:00 and 2


Both of the first two are semantic memory. One takes **four microseconds** and is exact; the other takes nearly two seconds, most of it an embedding call over the network, and returns a ranking.

The third block is the part that matters. There is no timezone anywhere in that journal, and the vector index answers anyway — *"Releases go out on Tuesdays"*, scored `0.5835`, which looks like a number you could trust. `03_operations` built a floor for exactly this, and a floor is a patch on a question that should never have been asked of this store.

So the type does not pick the store. **Nameability does**, which is the same criterion notebook 2 used all along — what will be asked of it:

| Type | The question it tends to get | Store |
|---|---|---|
| Episodic | *when*, *how many*, *which was last* | relational |
| Semantic, nameable | *what is X* | key–value |
| Semantic, not nameable | *what do I know about X* | vectors |
| Procedural | *how do I do X* | the prompt and the code; a table when it needs a history |

Read the middle two rows together: **semantic memory is not one storage decision.** "The user's timezone is Asia/Kolkata" and "checkout's pool was resized because the per-node share had fallen too low" are both facts, and they belong in different stores, because only one of them has a name you could ask for.

The type is a useful prior — it tells you which questions are *likely* — and it is not the decision. Which is why this notebook's real payoff is not the table above, but the branch it puts in the write path.

## Where the types actually diverge: the write path

Everything above is about *where* a memory goes. This is about what happens when it arrives, and it is the reason the three names are worth knowing at all.

`03_operations` built a write path that searches before it writes and then decides **add / update / ignore**. It exists so the same fact does not end up in the journal five times, and for facts it works — measured there at three correct decisions out of three.

Here is that same procedure, unchanged, looking at two deploy failures nine days apart. The dates are in the `at` column, where dates live, so the text of the two entries differs by one word:

In [12]:
from typing import Literal, Optional

from pydantic import BaseModel, Field, model_validator

from llm import extract


class WriteDecision(BaseModel):
    action: Literal["add", "update", "ignore"]
    target_id: Optional[int] = Field(default=None)
    reason: str

    @model_validator(mode="after")
    def update_needs_a_target(self):
        if self.action == "update" and self.target_id is None:
            raise ValueError("action 'update' requires target_id")
        if self.action != "update" and self.target_id is not None:
            raise ValueError(f"target_id is only meaningful for 'update', not '{self.action}'")
        return self


AS_A_FACT = """An agent has just concluded something. Decide what to do with it, given what it
already remembers.

New conclusion:
    {candidate}

Existing memories that resemble it:
{related}

  add     nothing remembered covers this
  update  one of them states the same thing with a value or detail this replaces -- name which
  ignore  one of them already says this

Return JSON matching the schema."""

NEW_EVENT = "Deploy of billing-api failed and was rolled back after twelve minutes."
REMEMBERED = {
    31: ("2026-08-03", "Deploy of billing-api failed and was rolled back after nine minutes."),
    11: ("2026-09-22", "Billing rolled back 2.9.4 after the error rate hit four percent."),
    18: ("2026-11-07", "The backfill was the cause; it holds a row lock per batch."),
}

plain = "\n".join(f"    #{i}  {t}" for i, (_, t) in REMEMBERED.items())

print("new event   [2026-08-12] ", NEW_EVENT)
print("remembered  [2026-08-03]  #31", REMEMBERED[31][1])
print("correct answer: add. Two deploys failed; neither replaces the other.\n")

verdicts = [extract(AS_A_FACT.format(candidate=NEW_EVENT, related=plain), WriteDecision)
            for _ in range(5)]
print("  ", "  ".join(f"{v.action}{'' if v.target_id is None else '#%d' % v.target_id}"
                       for v in verdicts))
print("   events destroyed:", sum(1 for v in verdicts if v.action != "add"), "of 5")

new event   [2026-08-12]  Deploy of billing-api failed and was rolled back after twelve minutes.
remembered  [2026-08-03]  #31 Deploy of billing-api failed and was rolled back after nine minutes.
correct answer: add. Two deploys failed; neither replaces the other.



   update#31  update#31  update#31  update#31  update#31
   events destroyed: 5 of 5


**Five out of five.** Every run decided that the 12th of August *updates* the 3rd of August, which in `03_operations`' write path means `journal.update(31, ...)` — the earlier deploy's text overwritten, the row reused, the failure on the 3rd gone with no error raised anywhere.

And the procedure is not malfunctioning. Asked *"does this new text state the same thing as one of these with a detail that replaces it?"*, the honest answer about two near-identical sentences is yes. The question was wrong, not the answer.

Notice also what made it certain rather than occasional. With the dates written into the prose — *"failed on 3 August"* against *"failed on 12 August"* — the same check gets it right about two times in three. The dates are not in the prose, because they are in a column, which is the correct place for them. **The one field that distinguishes the two events is the one field the duplicate check never sees.**

So the fix is not a better prompt for the same question. It is asking a different question, which requires knowing what kind of memory this is:

In [13]:
AS_AN_EVENT = """An agent has just recorded something. Decide what to do with it, given what
it already remembers.

This is an **event**: a thing that happened at a particular time. Two events that read alike
are still two events. Only use `update` or `ignore` if this is a correction to the SAME
occurrence, never because a similar thing happened before.

New event ({when}):
    {candidate}

Existing memories that resemble it, with the date each is about:
{related}

  add     this is a distinct occurrence
  update  this corrects the record of one of those same occurrences -- name which
  ignore  one of them already records this same occurrence

Return JSON matching the schema."""

dated = "\n".join(f"    #{i}  [{d}]  {t}" for i, (d, t) in REMEMBERED.items())

verdicts = [extract(AS_AN_EVENT.format(candidate=NEW_EVENT, related=dated, when="2026-08-12"),
                    WriteDecision)
            for _ in range(5)]
print("  ", "  ".join(f"{v.action}{'' if v.target_id is None else '#%d' % v.target_id}"
                       for v in verdicts))
print("   events destroyed:", sum(1 for v in verdicts if v.action != "add"), "of 5")

   add  add  add  add  add
   events destroyed: 0 of 5


Nothing clever happened there. Two changes: the date each memory is about went into the listing, and the prompt said *this is an event, and two events that read alike are still two events.*

Which is the chapter's own lesson again, from `03_operations`' two-stage recall: **whatever the decision depends on has to be in front of the thing deciding.** It was true of a model choosing which memory to open, and it is true of a model deciding whether a memory is a duplicate.

### Routing the write by type

Once the type is known, most of the decision disappears. An event never needs a duplicate check at all — it is a new occurrence by definition, and searching before writing it is a model call spent to be told `add`:

In [14]:
def remember(journal, kind, at, about, text, source="tool_output"):
    """Write a memory, taking the route its kind requires.

    event  -> append. Nothing to check: a new occurrence cannot be a duplicate of an old one.
    fact   -> search first, then add / update / ignore, because a fact can be superseded.
    """
    if kind == "event":
        return ("add", journal.write([(at, about, source, text)], now=TODAY)[0], "events append")

    related = [(eid, journal.get(eid)["text"])
               for _, eid, _ in journal.find_like(text, k=3)]
    block = "\n".join(f"    #{i}  {t}" for i, t in related)
    decision = extract(AS_A_FACT.format(candidate=text, related=block), WriteDecision)
    if decision.action == "add":
        return ("add", journal.write([(at, about, source, text)], now=TODAY)[0], decision.reason)
    if decision.action == "update":
        journal.update(decision.target_id, text, now=TODAY)
        return ("update", decision.target_id, decision.reason)
    return ("ignore", None, decision.reason)


before = len(history)
for at, text in [("2026-11-30", "Deploy of billing-api failed; rolled back after six minutes.")]:
    print(remember(history, "event", at, "billing-api", text))
print("history:", before, "->", len(history), "entries\n")

# remember what the store looked like first: the cleanup below should not depend on
# which branch the model took, only on what is actually there afterwards
before_ids = {r["id"] for r in memories.db.execute("SELECT id FROM entries")}

print(remember(memories, "fact", TODAY, "checkout-api",
               "Arun has lowered the backfill batch size cap to two hundred rows."))

('add', 5, 'events append')
history: 4 -> 5 entries



('update', 21, 'Existing memory #21 states a cap of 500 rows, but the new conclusion lowers it to 200 rows, so the memory should be updated.')


An event costs one insert and no model call. A fact costs a search and a decision, because it might be replacing something.

That is the practical return on the three names. Not a taxonomy to recite — a branch in the write path, where taking the wrong arm quietly deletes history that nothing will ever tell you is missing.

In [15]:
# housekeeping: the scratch history goes, and the fact written above is put back
for entry_id in {r["id"] for r in memories.db.execute("SELECT id FROM entries")} - before_ids:
    memories.forget(entry_id)              # whatever got appended, whichever branch ran
memories.update(21, "Arun wants the backfill batch size capped at five hundred rows from now on.",
                now="2026-11-20")
memories.db.execute("DELETE FROM accesses")
memories.db.commit()

org.run("MATCH (n:Thing {name: 'user'}) DETACH DELETE n")   # leave the graph as found

del history
shutil.rmtree(SCRATCH, ignore_errors=True)      # Chroma may hold the handle on Windows
print("graph   :", len(org), "edges")
org.close()

print("memories:", len(memories), "entries")
print("#21 restored:", memories.get(21)["text"])

graph   : 6 edges
memories: 24 entries
#21 restored: Arun wants the backfill batch size capped at five hundred rows from now on.


## What you built

✓ Separated the three types by **what makes each one wrong**, rather than by subject: a fact is superseded, an event cannot be, a rule is revoked — so **facts get replaced and events accumulate**

✓ Closed the usual list of memory types by crossing it with a second axis, lifetime, and found that `Scratchpad` and the saved conversation were never additional kinds — they are two cells in the table, which is why they needed no store

✓ Watched a vector index fail to say which of four events happened last, with **0.0037** between best and worst, and the middle two returned out of order — because "last" is a property of a column, not of meaning

✓ Answered the same question with `ORDER BY at DESC LIMIT 1`, and noted the one question similarity cannot be asked at all: **`COUNT`**

✓ Put a fact in a graph, watched it work, and left it out anyway — a graph earns its place when the answer is in the connections, and a value under a name has none to walk

✓ Found most procedural memory already written down in tool descriptions and the system prompt, with only the *learned* slice needing a store — plus the one thing a prompt cannot do, which is say what it used to be, so a changed rule wants an append-only table rather than an edit

✓ Found the reason a stored procedure goes missing: it is not hard to find, it is **outnumbered**. Three policies among twenty-four memories lost two of five situations to incident reports phrased like the questions — and that competition worsens every time the agent learns anything

✓ Fixed it by moving procedures out rather than ranking them better — a **skill catalogue**: one-line descriptions always in context, bodies loaded only when chosen. Fifteen selections, fifteen correct, with no ranking involved at all

✓ Named the reason it works, which is not the flattering one: not better wording, just four candidates instead of twenty-four

✓ Rejected the obvious type-to-store mapping where it is wrong: **semantic memory is not one storage decision.** A nameable fact is four microseconds and exact; an unnameable one is two seconds and ranked; and the nameable one asked of a vector index returns *"Releases go out on Tuesdays"* at `0.5835` from a journal containing no timezone at all

✓ Destroyed a run history with `03_operations`' own write path: **five times out of five**, a deploy failure on the 12th was recorded as an update to the one on the 3rd, silently

✓ Traced why it was certain rather than occasional — the dates live in a column, so the one field distinguishing the two events is the one field the duplicate check never sees

✓ Fixed it at **five out of five** by putting the dates in the listing and saying what kind of memory it was, which is the same lesson two-stage recall taught: whatever the decision depends on has to be in front of the thing deciding

✓ Routed the write by type, so an event costs one insert and no model call, and only a fact pays for a search

**Next:** `05_memory_in_the_loop.ipynb` — wiring all of it back into `02_agent_runtime`, which turns out to need no modification: a component is anything with `.run(conversation)`, so recall slots in beside `Observe`. Recall by injection before `Think`, write by tool — and the one thing genuinely missing, which is anywhere on `Conversation` to hang a store.